# 05 — The full chain: ewm-sm → Laya → Bonsai

**Question this notebook answers:** how do the three systems cooperate
on one turn, and what does each hop contribute?

```text
query → ewm-scene features → Laya typed decision → context proposal
      → Bonsai answer → tokenize → new frame → ewm-scene S(t) → next turn
```

This notebook runs a minimal controller loop with the raw clients and
prints every hop. Companion reference: `docs/INTERACTION_MAP.md` §7.


In [ ]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



In [ ]:
import json, os

WORK = os.path.join(PROJ, "work", "nb05")
os.makedirs(WORK, exist_ok=True)

ewm = EwmScene()
bonsai = BonsaiClient()
print("ewm-scene:", ewm.bin)
print("bonsai health:", bonsai.health())

RUN_CHAIN = bonsai.health()
if not RUN_CHAIN:
    print()
    print("Bonsai is DOWN — the cells below will print the requests they WOULD send.")
    print("Start it with ~/tools/Bonsai-demo scripts, then re-run this notebook.")


In [ ]:
# Start Laya once; reuse the daemon across turns.
laya = LayaDaemon()

union = os.path.join(WORK, "chain_union.jsonl")
with open(union, "w") as fh:
    fh.write("")

# controller state
seen = set()
comp_mem = []          # D-part novelty stream
frame_tids = []        # per-turn answer tids
mem_tids = []          # full materialized memory (last frame, restored order)
policy_cap = 24

def displacement(tids):
    out = []
    for t in tids:
        if t not in seen:
            out.append(t)
            seen.add(t)
    return out

def proposal(mode, cap):
    if mode == "full":
        return mem_tids[-cap:]
    if mode == "compact":
        return comp_mem[-cap:]
    if mode == "surprise":
        extra = []
        try:
            noe = ewm.noether(union)
            thr = {"dp": 6, "ind1": 0.5, "ind2": 0.5, "ind3": 0.5}
            flags = [False] * len(noe["dp"])
            for i in range(len(noe["dp"])):
                flags[i] = (noe["dp"][i] > thr["dp"] or noe["ind1"][i] > thr["ind1"]
                            or noe["ind3"][i] > thr["ind3"]
                            or (i >= 1 and noe["ind2"][i-1] < thr["ind2"]))
            for i, f in enumerate(flags):
                if f:
                    for idx in (i, i + 1):
                        if idx < len(frame_tids):
                            extra.extend(frame_tids[idx])
        except Exception:
            pass
        raw = comp_mem[-cap:] + extra
        out = []
        for t in raw:
            if not out or out[-1] != t:
                out.append(t)
        return out[-cap:]
    return []

def features_for(query):
    uni = ewm.ingest(union)
    noe = ewm.noether(union)
    return {
        "turn": len(uni["frames"]),
        "union_pop": uni["frames"][-1]["pop"] if uni["frames"] else 0,
        "full_memory_tids": len(mem_tids),
        "novelty_tids": len(comp_mem),
        "last_dp": noe["dp"][-1] if noe.get("dp") else 0,
    }

def one_turn(query):
    global mem_tids
    # hop 1: ewm-scene features
    feats = features_for(query)
    state = LayaDaemon.prose_state(query, memory_tokens=feats["full_memory_tids"])

    # hop 2: Laya decides the context policy
    d = laya.route(
        state,
        options={
            "full": "keep the full materialized memory as the context",
            "compact": "keep only new, previously unseen tids as the context",
            "surprise": "keep only structurally surprising content",
        },
        instructions="How should the next context be built for the base model?",
        extra_noul={"keep_short": "The context should be kept short"},
        verbose=False,
    )
    mode = d["decision"] if d["decision"] in ("full", "compact", "surprise") else "full"
    cap = 8 if d["aux"]["keep_short"] > 0.5 else policy_cap

    # hop 3: context proposal -> detokenize -> prompt
    prefix_tids = proposal(mode, cap)
    if prefix_tids:
        prefix_text = bonsai.detokenize([int(t[3:]) for t in prefix_tids])
        prompt = f"Context memory ({mode}):\n{prefix_text}\n\nQuery: {query}"
    else:
        prompt = query

    # hop 4: Bonsai answers (or prints the request if the server is down)
    if RUN_CHAIN:
        full = bonsai.chat_full(prompt, max_tokens=128, temperature=0.0)
        answer = full["content"]
        prompt_tokens = int(full["usage"].get("prompt_tokens", 0))
    else:
        print("── would POST /v1/chat/completions ──")
        print(json.dumps({"messages": [{"role": "user", "content": prompt}],
                          "max_tokens": 128, "temperature": 0.0, "stream": False}, indent=2))
        answer = "(bonsai down)"
        prompt_tokens = len(prompt.split())

    # hop 5: tokenize the answer back into tid frames
    if RUN_CHAIN:
        tids = [f"tid{i}" for i in bonsai.tokenize(answer)]
    else:
        tids = [f"tid{1000 + j}" for j in range(3)]

    # hop 6: ingest into the lattice
    comp_mem.extend(displacement(tids))
    frame_tids.append(tids)
    with open(union, "a") as fh:
        fh.write(json.dumps({"id": feats["turn"] + 1, "tokens": tids}) + "\n")
    try:
        mat = ewm.materialize(union)
        mem_tids = [t for t in mat["frames"][-1]["ordered"]]
    except Exception:
        mem_tids = list(tids)

    uni = ewm.ingest(union)
    return {
        "turn": len(uni["frames"]),
        "query": query,
        "mode": mode,
        "cap": cap,
        "prefix_tids": len(prefix_tids),
        "prompt_tokens": prompt_tokens,
        "answer": answer,
        "pop": uni["frames"][-1]["pop"],
        "laya_conf": round(d["confidence"], 3),
        "keep_short": round(d["aux"]["keep_short"], 3),
    }


## Run three turns and watch the chain


In [ ]:
rows = []
for q in [
    "What is the capital of France? Answer in one short sentence.",
    "Explain gravity in one sentence.",
    "Write a haiku about rain.",
]:
    rows.append(one_turn(q))

print(f"{'t':>2} {'mode':9s} {'cap':>3} {'prefix':>6} {'prompt_tok':>10} {'pop':>4} {'conf':>6} {'short':>6}  answer")
for r in rows:
    print(f"{r['turn']:>2} {r['mode']:9s} {r['cap']:>3} {r['prefix_tids']:>6} "
          f"{r['prompt_tokens']:>10} {r['pop']:>4} {r['laya_conf']:>6} {r['keep_short']:>6}  {r['answer'][:48]!r}")


## What to observe

- `prefix_tids` grows after turn 1 — Bonsai starts receiving a context
  prefix built from the lattice.
- `mode`/`cap` change per turn — Laya's typed decision, not a fixed rule.
- `prompt_tokens` is Bonsai's own accounting of the prefix + query.
- `pop` is the lattice's accounting of the last frame (≈ 3× distinct
  tids).
- When `keep_short > 0.5` the controller drops the cap to 8.
- **Bonsai is a reasoning model.** With a small `max_tokens`, the
  reasoning trace can consume the whole budget and `content` comes back
  empty (the lattice then ingests an empty frame, `pop=0`). This notebook
  uses `max_tokens=128`; the server log suggests `--reasoning-preserve`
  as the long-term fix.


In [ ]:
laya.close()
print('daemon closed')
